# NC vs Patient MSN 网络分析

仅针对 **MSN 网络边水平**，不涉及拓扑特征。流程：

1. **数据准备**：加载 NC + Patient 的 MSN 矩阵（148×148 → N×10878）、协变量（age, sex, eTIV）、group（0=NC, 1=Patient）
2. **逐边 GLM**：`edge_weight ~ group + age + sex + eTIV`，得到 10878 个 t 与 p
3. **FDR 校正**：Benjamini–Hochberg，得到显著边
4. **NBS**：阈值 + 连通子网络 + permutation，得到显著子网络
5. **结果导出与简单可视化**

## 1. 环境与路径

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"
HEALTH_MSN_DIR = RESULTS_ROOT / "health_msn_matrices"
PATIENT_MSN_DIR = RESULTS_ROOT / "patient_msn_matrices"
HEALTH_INFO_PATH = DATA_ROOT / "health_info.csv"
PATIENT_INFO_PATH = DATA_ROOT / "patient_info.csv"
OUT_DIR = RESULTS_ROOT / "msn_net_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根:", ROOT)

## 2. 数据准备（Network feature matrix）

构建 **N × 10878** 边矩阵，以及协变量、group。

In [ ]:
import numpy as np
from msn_stats.data import load_nc_patient_data, build_design_nc_patient, N_NODES

edge_matrix, group, age, sex, eTIV, subject_ids = load_nc_patient_data(
    HEALTH_MSN_DIR,
    PATIENT_MSN_DIR,
    HEALTH_INFO_PATH,
    PATIENT_INFO_PATH,
    n_nodes=N_NODES,
    use_processed=True,
)

#若下方“全局强度”组间差异大且逐边几乎全显著，可改为控制 global_strength：
global_strength = edge_matrix.mean(axis=1)
design = build_design_nc_patient(group, age, sex, eTIV, global_strength=global_strength)
contrast = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0])

#design = build_design_nc_patient(group, age, sex, eTIV)
#contrast = np.array([0.0, 1.0, 0.0, 0.0, 0.0])

print("边矩阵形状:", edge_matrix.shape)
print("设计矩阵形状:", design.shape)
print("NC 人数:", (group == 0).sum(), "Patient 人数:", (group == 1).sum())

In [ ]:
# 检查“全局强度”（每人边权均值）：若组间差异很大，逐边检验会几乎全显著，建议将 global_strength 加入设计矩阵
global_strength = edge_matrix.mean(axis=1)
nc_gs = global_strength[group == 0]
pt_gs = global_strength[group == 1]
print("每被试边权均值: NC = {:.4f} ± {:.4f}, Patient = {:.4f} ± {:.4f}".format(nc_gs.mean(), nc_gs.std(), pt_gs.mean(), pt_gs.std()))
print("组间差异 (Patient - NC) = {:.4f}".format(pt_gs.mean() - nc_gs.mean()))

均值：patient的均值约为NC的2.3倍，组间差0.0117相对NC均值约+133%。

变异：Patient 标准差（0.0091）也明显大于 NC（0.0039），组内变异更大。

也就是说，患者组在「每条边平均强度」这个整体水平上系统性高于健康组，且组间差异在数值和比例上都不小。

若不控制整体强度，很多边会仅仅因为「患者整体连接更高」而在一维 t 检验或简单 GLM 里显著，难以区分。

把 global_strength 作为协变量加入设计矩阵后，回归的是「在相同整体强度下」的组效应，得到的显著边更接近边特异的组间差异，而不是被整体强度差异“摊平”到所有边上。

In [ ]:
edge_var = np.var(edge_matrix, axis=0, ddof=1)
mask_keep = edge_var > 1e-8   # 阈值可调
print("keep edges:", mask_keep.sum(), "drop:", (~mask_keep).sum())
edge_matrix2 = edge_matrix[:, mask_keep]
# 用 edge_matrix2 重新跑 edgewise_glm_twogroup

In [ ]:
np.linalg.cond(design)
np.linalg.cond(design.T @ design)

In [ ]:
for i in range(design.shape[1]):
    print(i, design[:,i].mean(), design[:,i].std(), design[:,i].min(), design[:,i].max())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_rank, cond

# 假设你已有 edge_matrix (n, E) 和 design (n, p)
n, E = edge_matrix.shape
n, p = design.shape

# 1) 检查 X'X 的条件数 / 秩
XtX = design.T @ design
print("design shape:", design.shape)
print("design rank:", matrix_rank(design))
print("XtX condition number:", cond(XtX))

# 2) 检查 MSE 的分布（使用你内部 _ols_fit 或自己计算）
def ols_mse(X, Y):
    n, p = X.shape
    B = np.linalg.inv(X.T @ X) @ (X.T @ Y)
    R = Y - X @ B
    df = n - p
    MSE = np.sum(R * R, axis=0) / max(df, 1)
    return B, R, MSE

B, R, MSE = ols_mse(design, edge_matrix)
print("MSE: min, 1%, 5%, 10%, 25%, 50%, 75%, 90%, 99%, max:")
print(np.percentile(MSE, [0,1,5,10,25,50,75,90,99,100]))

# 3) 看有多少 MSE 接近 0
tiny_thresh = 1e-10
print("count MSE <= 1e-10:", np.sum(MSE <= tiny_thresh))
print("count MSE <= 1e-8:", np.sum(MSE <= 1e-8))
print("count MSE <= 1e-6:", np.sum(MSE <= 1e-6))
print("fraction <=1e-6:", np.mean(MSE <= 1e-6))

# 4) 检查 effect (beta for group) 的分布与 effect / sqrt(MSE) 的关系
# 需要知道你 group 列索引 c (1)
group_col = 1
# compute iXtX and c_iXtX_c
iXtX = np.linalg.inv(XtX)
c = np.zeros(p); c[group_col] = 1.0
c_iXtX_c = float(c @ iXtX @ c)
print("c_iXtX_c:", c_iXtX_c)

effect = (c @ B)  # length E
print("effect percentiles (group beta):", np.percentile(effect, [0,1,5,10,25,50,75,90,99,100]))

# approximate se per edge (without mse floor)
se = np.sqrt(np.maximum(c_iXtX_c * MSE, 1e-20))
tvals = effect / se
print("|t| percentiles:", np.percentile(np.abs(tvals), [0,1,5,10,25,50,75,90,99,100]))

# 5) 检查边的原始方差（样本方差 across subjects）
edge_var = np.var(edge_matrix, axis=0, ddof=1)
print("edge_var percentiles:", np.percentile(edge_var, [0,1,5,10,25,50,75,90,99,100]))

# 6) 随机抽取几条边，画残差直方图与残差时间序列
import random
indices = np.concatenate([np.random.choice(E, size=3, replace=False), np.argsort(MSE)[:3]])
indices = np.unique(indices)
for idx in indices:
    r = R[:, idx]
    print("edge", idx, "MSE:", MSE[idx], "var(edge):", edge_var[idx], "effect:", effect[idx])
    plt.figure(figsize=(6,2))
    plt.subplot(1,2,1)
    plt.hist(r, bins=30)
    plt.title(f"resid hist edge {idx}")
    plt.subplot(1,2,2)
    plt.plot(r, '.-')
    plt.title("resid series")
    plt.tight_layout()
    plt.show()

### 残差图怎么读

- **左图（残差直方图）**：该边在所有被试上的拟合残差分布。期望大致**钟形、中心在 0**；若严重偏态或双峰，可留意该边或模型假设。
- **右图（残差序列）**：同一批残差按**被试顺序**连线。期望在 0 附近**随机波动**；若出现明显上升/下降趋势或成段偏高/偏低，可能存在未纳入的与顺序相关的因素（如扫描顺序、中心）。
- 这里画的是：随机 3 条边 + MSE 最小的 3 条边，用于抽查残差是否正常。

## 3. 逐边 GLM 与 FDR 校正

模型：`edge ~ group + age + sex + eTIV`；FDR α=0.05。

In [ ]:
from msn_stats.edgewise import edgewise_glm_twogroup
from msn_stats.fdr import fdr_correct

FDR_ALPHA = 0.01

# 默认 mse_floor_percentile=10；若仍几乎全显著可试 mse_floor_percentile=15
t_obs, p_obs, beta_group = edgewise_glm_twogroup(edge_matrix, design, group_col=1)
rejected, p_fdr = fdr_correct(p_obs, alpha=FDR_ALPHA)
n_sig_edges = rejected.sum()

print("逐边 GLM 完成。FDR 显著边数:", n_sig_edges, "/", len(p_obs))

# 诊断：若之前出现 10878/10878 全显著，多为残差方差(MSE)过小导致 t 爆炸；已用 MSE 下界修复
print("p 值分布: min={:.2e}, 1%={:.2e}, 50%={:.2e}, 99%={:.2e}".format(
    np.min(p_obs), np.percentile(p_obs, 1), np.median(p_obs), np.percentile(p_obs, 99)))
print("|t| 分布: 50%={:.2f}, 99%={:.2f}".format(np.median(np.abs(t_obs)), np.percentile(np.abs(t_obs), 99)))

## 6. 组平均网络（辅助解释）

分别计算 NC 与 Patient 的 mean MSN，便于可视化对比。

In [ ]:
nc_mask = group == 0
pt_mask = group == 1
mean_nc = edge_matrix[nc_mask].mean(axis=0)
mean_pt = edge_matrix[pt_mask].mean(axis=0)
np.save(OUT_DIR / "mean_edge_nc.npy", mean_nc)
np.save(OUT_DIR / "mean_edge_patient.npy", mean_pt)
print("组平均边权重已保存。")